# Practical PyTorch · Phase II, Part 9 (Capstone) — Companion Notebook

This goes with **"The Capstone: Fine-Tune a Model and Put It on the Hub"**. Run the cells top to bottom (Shift+Enter).

**Turn the GPU on:** Runtime → Change runtime type → Hardware accelerator → **GPU**. On CPU this still works — it's just slower.

The dataset is kept small on purpose so the whole thing finishes in a few minutes.

## 0. Install the Hugging Face stack

Colab ships with PyTorch, but not the rest. This is the only install you need.

In [ ]:
!pip install -q transformers datasets evaluate

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. The task and a small slice of data

We're fine-tuning a tiny BERT (`distilbert-base-uncased`) to tell positive movie reviews from negative ones, using the `imdb` dataset. To keep the run honest about time, we take a **slice** — 2,000 reviews to train on, 1,000 to evaluate — not the full 25,000.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")
train_ds = dataset["train"].shuffle(seed=42).select(range(2000))
eval_ds  = dataset["test"].shuffle(seed=42).select(range(1000))

print(train_ds)
print("\nOne example:")
print("label:", train_ds[0]["label"], "(0 = negative, 1 = positive)")
print("text :", train_ds[0]["text"][:200], "...")

## 2. Tokenize the text

Models eat numbers, not strings. The tokenizer turns each review into token IDs. `truncation=True` clips reviews to the model's max length so nothing overflows.

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

train_ds = train_ds.map(tokenize, batched=True)
eval_ds  = eval_ds.map(tokenize, batched=True)

print("Done. Each example now also has input_ids + attention_mask.")

## 3. The model: pretrained backbone, fresh 2-class head

`AutoModelForSequenceClassification` keeps DistilBERT's pretrained language understanding and bolts a brand-new, untrained head on top, sized for our two labels. We also set `id2label` so the model returns human-readable labels later instead of `LABEL_0` / `LABEL_1`.

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# The warning about uninitialized 'classifier' weights is expected —
# that's the new head we're about to train.
print("Model ready.")

## 4. How we'll score it: compute_metrics

From Part 6 — turn raw predictions into accuracy. The `Trainer` calls this automatically on the held-out eval set.

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

## 5. Fine-tune with the Trainer

This is the Part 5 move. The data collator pads each batch to the same length on the fly. One epoch over 2,000 examples is enough to get a clearly-useful model in a few minutes on a GPU.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="distilbert-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    logging_steps=25,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Evaluate it honestly

"The code ran" is not "the model is good." This reports accuracy on the 1,000 held-out reviews the model never trained on — expect something comfortably in the high 80s.

In [ ]:
metrics = trainer.evaluate()
print(metrics)

## 7. Log in to the Hub

Pushing needs a **write** token: create one at https://huggingface.co/settings/tokens (give it write access).

Two ways to log in — use whichever you like:

- **Token widget:** run `login()` and paste your token into the box.
- **Colab secret (cleaner):** add a secret named `HF_TOKEN` (the key icon in the left sidebar), then use the commented-out lines below — no pasting into a cell.

Never paste a token into a cell you plan to share.

In [ ]:
from huggingface_hub import login

# Option A — paste your token into the widget:
login()

# Option B — Colab secret named HF_TOKEN (uncomment to use):
# from google.colab import userdata
# login(userdata.get("HF_TOKEN"))

## 8. Push the model and tokenizer

**Replace `your-username` with your real Hugging Face username** — it's a placeholder.

Because we trained with the `Trainer`, one call pushes the weights, the tokenizer, the config, and a starter model card. It creates a repo named after `output_dir` under your account.

In [ ]:
# The Trainer way — pushes everything to <your-username>/distilbert-imdb:
trainer.push_to_hub()

# If you'd trained WITHOUT the Trainer, you'd push the two pieces by hand
# (both must go to the SAME repo):
#
# model.push_to_hub("your-username/distilbert-imdb")
# tokenizer.push_to_hub("your-username/distilbert-imdb")

## 9. Load it back and run it — the proof

Forget the notebook state. We load the model **purely by its Hub name**, exactly the way a stranger would, and run it through a `pipeline`. No training code in sight.

**Replace `your-username`** with yours. (First load downloads from the Hub, so it's not instant.)

In [ ]:
from transformers import pipeline

# ⬇️ Replace your-username with your actual Hugging Face username:
repo_id = "your-username/distilbert-imdb"

pipe = pipeline("text-classification", model=repo_id)

print(pipe("This was the most fun I've had at the movies all year."))
print(pipe("Two hours of my life I will never get back."))

That's the whole loop closed: raw text → a model **you fine-tuned and shipped** → loaded from the open internet by name. Because we set `id2label` earlier, it answers `POSITIVE` / `NEGATIVE` instead of `LABEL_1` / `LABEL_0`.

**You can fine-tune a model and ship it.** That's Phase II. Go own one.